In [1]:
%reload_ext autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import IPython.display as ipd
import whisper
import sys
sys.path.append("/home/romolo/VT1/coqui-tts")
from model_conf import ModelPaths, load_tts_and_trainer
import os
from development.utils import iterative_segment_refinement,denoised_temp_file
from jiwer import wer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import torch
from huggingface_hub import hf_hub_download

/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [3]:


# automatically checks for cached file, optionally set `cache_dir` location
model_file = hf_hub_download(repo_id='Jenthe/ECAPA2', filename='ecapa2.pt', cache_dir=None)

In [4]:
ecapa2 = torch.jit.load(model_file, map_location='cuda')

In [5]:
paths = ModelPaths()
tts, model, train_model, config = load_tts_and_trainer(paths)

 > Using model: xtts
>> DVAE weights restored from: /home/romolo/VT1/coqui-tts/XTTS_v2.0_original_model_files/dvae.pth


In [6]:
asr_model = whisper.load_model('base')

In [7]:
ref_samples = os.listdir("/home/romolo/VT1/coqui-tts/test_data/Dataset/references/26/")
ref_samples = ["/home/romolo/VT1/coqui-tts/test_data/Dataset/references/26/" + i for i in ref_samples]
orig_target_sample = "/home/romolo/VT1/coqui-tts/data/target.wav"

In [8]:
asr_model.transcribe(orig_target_sample)['text']

' No, sir, I did not mean it so, and I am very, very sorry. Dear pop-up, please forgive me, and I will try never to forget again." I think you disobeyed in another matter, he said.'

In [9]:
# Call the function with your existing variables
with denoised_temp_file(orig_target_sample) as denoised_path:
    final_audio = iterative_segment_refinement(
        model=model,                                    # Your XTTS model
        target_speaker=denoised_path,              # Path to target speaker audio
        ref_speaker=ref_samples,                        # List of reference samples
        max_conditioning_length=config.model_args.max_conditioning_length,
        min_conditioning_length=config.model_args.min_conditioning_length,
        train_model=train_model,                        # Your trained model
        asr_model=asr_model,                           # Your ASR model
        ecapa_model=ecapa2,                            # Your ECAPA model
        lang='en',                                     # Language
        threshold=0.60,                                 # Quality threshold (adjust as needed)
        max_attempts=10,
        regenerate=True
    )

/home/romolo/VT1/coqui-tts/development/utils.py:315: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target_audio_16k = resample_audio_16k(torch.tensor(target_audio), orig_freq=24000, new_freq=16000)
/home/romolo/VT1/coqui-tts/development/utils.py:317: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ref_audio_16k = resample_audio_16k(torch.tensor(ref_audio), orig_freq=24000, new_freq=16000)


In [10]:
ipd.Audio(final_audio, rate=24000)

In [12]:
final_audio = iterative_segment_refinement(
        model=model,                                    # Your XTTS model
        target_speaker=orig_target_sample,              # Path to target speaker audio
        ref_speaker=ref_samples,                        # List of reference samples
        max_conditioning_length=config.model_args.max_conditioning_length,
        min_conditioning_length=config.model_args.min_conditioning_length,
        train_model=train_model,                        # Your trained model
        asr_model=asr_model,                           # Your ASR model
        ecapa_model=ecapa2,                            # Your ECAPA model
        lang='en',                                     # Language
        threshold=0.60,                                 # Quality threshold (adjust as needed)
        max_attempts=10,
        regenerate=True
    )

/home/romolo/VT1/coqui-tts/development/utils.py:315: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target_audio_16k = resample_audio_16k(torch.tensor(target_audio), orig_freq=24000, new_freq=16000)
/home/romolo/VT1/coqui-tts/development/utils.py:317: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ref_audio_16k = resample_audio_16k(torch.tensor(ref_audio), orig_freq=24000, new_freq=16000)


In [ ]:
ipd.Audio(final_audio, rate=24000)